# Task 1.1: Core Contribution / Architecture
## Paper: Training and Testing Low-degree Polynomial Data Mappings via Linear SVM
**Authors:** Yin-Wen Chang, Cho-Jui Hsieh, Kai-Wei Chang, Michael Ringgaard, Chih-Jen Lin (2010)

---

## Step-by-Step Method Description

### Step 1: Problem Setup — Why Kernel SVM is Expensive
- The starting point is standard nonlinear SVM. We have training data $\{(\mathbf{x}_i, y_i)\}$ and we want to classify using a polynomial kernel $K(\mathbf{x}_i, \mathbf{x}_j) = (\gamma \mathbf{x}_i^T \mathbf{x}_j + r)^d$. The classic approach solves this in the dual using something like LIBSVM, but every training iteration needs kernel evaluations costing $O(l \cdot \bar{n})$, where $l$ is the dataset size and $\bar{n}$ is the average nonzero features per instance. For large datasets, this gets really slow.
- **Paper Reference:** Equation (1) for the SVM primal; Section 2 for kernel definitions.
- **Purpose:** Sets up the motivation — kernel SVM works well but the cost doesn't scale.

### Step 2: The Key Idea — Explicit Polynomial Mapping
- Instead of implicitly using the kernel trick, the paper says: just compute the polynomial features directly. For a degree-2 kernel with $\gamma=1, r=1$, this mapping looks like:

$$\phi(\mathbf{x}) = [1,\; \sqrt{2}\,x_1, \ldots, \sqrt{2}\,x_n,\; x_1^2, \ldots, x_n^2,\; \sqrt{2}\,x_1 x_2, \ldots, \sqrt{2}\,x_{n-1}x_n]$$

  So you get a constant term, scaled linear features, squared features, and pairwise interaction terms. The scaling factors ($\sqrt{2}$, etc.) ensure that $\phi(\mathbf{x})^T \phi(\mathbf{x}') = (\mathbf{x}^T\mathbf{x}' + 1)^2$, which exactly reproduces the kernel value.
- **Paper Reference:** Equation (5) in Section 3.1. The general form with $\gamma$ and $r$ parameters is also in Eq. (5).
- **Purpose:** This is the core idea of the paper — by making the mapping explicit, we convert the nonlinear SVM problem into a linear one, letting us use much faster solvers.

### Step 3: Train a Linear SVM on the Mapped Data
- Once we have $\phi(\mathbf{x}_i)$ for all training instances, we just train a standard linear SVM (using LIBLINEAR) on this new representation. The optimization is the usual L2-regularised problem:

$$\min_{\mathbf{w}} \; \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^l \max(1 - y_i \mathbf{w}^T\phi(\mathbf{x}_i),\, 0)$$

  The important thing is that each iteration now costs $O(\hat{n})$ per instance, where $\hat{n}$ is the number of nonzero elements in $\phi(\mathbf{x})$. For sparse data, $\hat{n} \approx \bar{n}^2/2$ (Equation 10), which is way smaller than the kernel SVM's $O(l \cdot \bar{n})$ when the dataset is large.
- **Paper Reference:** Equation (1); Section 3.2 for cost analysis, Equations (7)–(9).
- **Purpose:** This is where the speedup happens — solving in the primal on mapped features is much cheaper than solving the dual with kernel evaluations.

### Step 4: Sparse Storage of the Weight Vector
- There's a practical problem: the mapped space can be huge. For $n = 47{,}236$ features (rcv1 dataset), the degree-2 mapped space has over a billion dimensions. You obviously can't store $\mathbf{w}$ as a dense vector. Fortunately, because the training data is sparse, most components of $\mathbf{w}$ never get updated and stay at zero. So the paper stores $\mathbf{w}$ using a hash map — only keeping track of the nonzero entries, indexed by feature pairs $(i,j)$.
- **Paper Reference:** Section 3.4, around Equation (11). They also derive an estimate for how many entries of $\mathbf{w}$ will be nonzero.
- **Purpose:** Without this, the method would be impractical for high-dimensional data. Sparse storage is what makes it actually work on real text classification problems.

### Step 5: Fast Prediction
- At test time, predicting a label is just: $\hat{y} = \text{sign}(\mathbf{w}^T \phi(\mathbf{x}) + b)$. This is a single dot product, costing $O(\hat{n}_t)$ — the number of nonzero mapped features in the test instance. Compare this to kernel SVM, where prediction requires summing over all support vectors: $\sum_{i \in SV} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x})$, costing $O(|SV| \cdot \bar{n}_t)$. Since the number of support vectors can be a large fraction of the training set, the explicit mapping gives a big speedup at test time too.
- **Paper Reference:** Section 3.2, testing cost analysis and Table 1.
- **Purpose:** Faster prediction is a practical bonus — the method doesn't just train faster, it also deploys faster.

### Step 6: Extensions — Higher Degrees and RBF Approximation
- The same idea works for degree-3 polynomials, though the mapped dimension grows as $\binom{n+d}{d}$, so you need to be more careful about memory. More interestingly, the paper shows you can approximate the RBF kernel by using a truncated Taylor expansion — basically expressing $e^{-\gamma\|\mathbf{x}-\mathbf{x}'\|^2}$ as a sum of polynomial terms and then using explicit mappings for each term.
- **Paper Reference:** Section 3.5 for the RBF approximation idea; Table 2 for mapped feature dimensions at different degrees.
- **Purpose:** Shows the approach isn't limited to degree-2 — it's a general framework, and even RBF-like behaviour can be approximated this way.

### Step 7: On-the-Fly Computation of $\phi(\mathbf{x})$
- Storing the entire mapped dataset would need $O(l \cdot \hat{n})$ memory, which is too much for large datasets. Instead, the paper computes $\phi(\mathbf{x}_i)$ on the fly during each training iteration and immediately updates $\mathbf{w}$, without ever materialising the full mapped dataset. This brings memory down to $O(\text{nnz}(\mathbf{w}) + l \cdot \bar{n})$ — just the sparse weight vector plus the original data.
- **Paper Reference:** Section 3.4, discussion on generating $\phi(\mathbf{x})$ on the fly.
- **Purpose:** Makes the method practical for large-scale problems where the mapped data simply wouldn't fit in memory.

---
## Final Summary

This paper tackles the computational bottleneck of nonlinear SVM — specifically, the expensive kernel evaluations during training and testing. The authors' approach is to explicitly compute the degree-2 polynomial feature mapping and then train a fast linear SVM (LIBLINEAR) on the result. For sparse data, this reduces the per-iteration training cost from $O(l \cdot \bar{n})$ to $O(\hat{n}) \approx O(\bar{n}^2/2)$, which is dramatically cheaper when the dataset is large relative to feature density. The authors claim their method is better than existing kernel SVM solvers because it achieves comparable accuracy with speedups of 4× to over 4,600× on standard benchmarks, while also making prediction faster and more memory-efficient.

**Paper References:** Sections 3.1–3.4 for the method; Section 4, Tables 3–7 and Figures 2–3 for experiments.